# 中证800 V57：随机组合基准 + FP/FN 错例诊断

## 实验目的

V57 不训练新模型，也不改 V46/V56 的模型结构。它只解决一个评价问题：

> 当前模型每个月实际选出的 target top6，放在真实交易场景里，相比“同股票池、同组合约束下随机抽 6 只”到底有多强？

同时做错例归因：

- **FN_top6**：真实未来 top6 但模型没有选中，分析为什么漏掉。
- **FP_top6**：模型选中但不在真实未来 top6，分析误选。
- **FP_not_top20**：模型选中且不在真实未来 top20，作为更严重误选。

这个 notebook 贴近实际应用：月频、CSI800、固定 V56 模型银行、target_top6/industry_top6 组合口径、未来 alpha_1m 标签。

In [ ]:
import os
import gc
import json
import pickle
import zipfile
import tempfile
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 240)


# tqdm is optional in JoinQuant/local notebooks. Fallback keeps long cells visibly moving.
try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", every=None):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc)
    if every is None:
        every = max(1, int((total or 100) / 20))

    def _gen():
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                if total is None:
                    print("%s %s" % (desc, i))
                else:
                    print("%s %s/%s" % (desc, i, total))
            yield item
    return _gen()

# =========================
# Config
# =========================
DATA_PATH = "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"
V56_OUTPUT_DIR = "csi800_ml_v56_v46_model_bank_outputs"
V56_OUTPUT_ZIP = "csi800_ml_v56_v46_model_bank_outputs.zip"
OUT_DIR = "csi800_ml_v57_random_baseline_fp_fn_outputs"

TARGET_COL = "alpha_1m"
STOCK_COL = "stock"
DATE_COL = "rebalance_date"
INDUSTRY_COL = "industry_bucket"
TARGET_PROFILE = "industry_top6"
TARGET_N = 6
REALIZED_TOP_N = 6
REALIZED_TOP20_N = 20
RANDOM_SIM_N = 3000
RANDOM_SEED = 42

# 空列表表示读取 V56 manifest 里的全部模型。常用主测：2019_2023 / 2019_2024。
MODEL_TAGS = ["2019_2023", "2019_2024"]

# 错例归因只看模型 gain 最高的前 N 个特征，避免解释变成噪音列表。
ATTR_TOP_FEATURE_N = 15
CASE_EXPORT_TOP_N_PER_MONTH = 20

os.makedirs(OUT_DIR, exist_ok=True)
print("output dir:", OUT_DIR)

## 数据读取

输入有两类：

1. `DATA_PATH`：V56 训练/评估用的月度样本表，必须包含 `stock/rebalance_date/alpha_1m` 和模型特征列。
2. `V56_OUTPUT_DIR` 或 `V56_OUTPUT_ZIP`：V56 导出的模型银行，包含 pkl、manifest、monthly。

如果本地没有解压目录但有 zip，本 cell 会自动解压到临时目录读取。

In [ ]:
def first_existing_path(paths):
    for path in paths:
        if path and os.path.exists(path):
            return path
    return None


def safe_to_datetime(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col])
    return out


def load_dataset(path):
    if not os.path.exists(path):
        raise IOError("data csv not found: " + path)
    df = pd.read_csv(path)
    df = safe_to_datetime(df, ["rebalance_date", "feature_date", "next_date"])
    if STOCK_COL not in df.columns:
        for alt in ["code", "security", "order_book_id"]:
            if alt in df.columns:
                df = df.rename(columns={alt: STOCK_COL})
                break
    if TARGET_COL not in df.columns:
        if "raw_return_1m" in df.columns and "benchmark_csi800_1m" in df.columns:
            df[TARGET_COL] = df["raw_return_1m"] - df["benchmark_csi800_1m"]
        else:
            raise ValueError("target column not found: " + TARGET_COL)
    if INDUSTRY_COL not in df.columns:
        df[INDUSTRY_COL] = "UNKNOWN"
    need = [STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError("dataset missing columns: " + ",".join(missing))
    return df


def resolve_v56_output_dir(output_dir, output_zip):
    if os.path.isdir(output_dir):
        return output_dir
    if os.path.exists(output_zip):
        tmp = tempfile.mkdtemp(prefix="v56_outputs_")
        with zipfile.ZipFile(output_zip) as z:
            z.extractall(tmp)
        print("extracted V56 zip to:", tmp)
        return tmp
    raise IOError("V56 output dir/zip not found: " + output_dir + " or " + output_zip)


def load_v56_outputs(v56_dir):
    manifest_path = os.path.join(v56_dir, "v56_model_bank_export_manifest.csv")
    monthly_path = os.path.join(v56_dir, "v56_model_bank_monthly.csv")
    if not os.path.exists(manifest_path):
        raise IOError("manifest not found: " + manifest_path)
    if not os.path.exists(monthly_path):
        raise IOError("monthly not found: " + monthly_path)
    manifest = pd.read_csv(manifest_path)
    monthly = pd.read_csv(monthly_path)
    manifest = safe_to_datetime(manifest, ["train_start", "train_end", "test_start"])
    monthly = safe_to_datetime(monthly, ["rebalance_date", "train_end", "test_start"])
    if MODEL_TAGS:
        manifest = manifest[manifest["tag"].astype(str).isin([str(x) for x in MODEL_TAGS])].copy()
        monthly = monthly[monthly["tag"].astype(str).isin([str(x) for x in MODEL_TAGS])].copy()
    monthly = monthly[monthly["profile"].astype(str) == TARGET_PROFILE].copy()
    if manifest.empty:
        raise ValueError("empty manifest after MODEL_TAGS filter")
    if monthly.empty:
        raise ValueError("empty monthly target rows after profile/tag filter")
    return manifest, monthly


def parse_targets(x):
    if pd.isnull(x):
        return []
    return [s.strip() for s in str(x).split(",") if s.strip()]


def load_bundle(v56_dir, manifest_row):
    model_file = str(manifest_row["model_file"])
    candidates = [
        os.path.join(v56_dir, model_file),
        str(manifest_row.get("model_path", "")),
        os.path.join(V56_OUTPUT_DIR, model_file),
    ]
    model_path = first_existing_path(candidates)
    if model_path is None:
        raise IOError("model pkl not found for " + model_file)
    with open(model_path, "rb") as f:
        bundle = pickle.load(f)
    if not isinstance(bundle, dict):
        raise ValueError("bundle should be dict: " + model_path)
    return bundle, model_path


df_all = load_dataset(DATA_PATH)
v56_dir = resolve_v56_output_dir(V56_OUTPUT_DIR, V56_OUTPUT_ZIP)
manifest_df, monthly_targets_df = load_v56_outputs(v56_dir)

print("data:", df_all.shape)
print(df_all[[DATE_COL]].agg(["min", "max"]))
print("v56 dir:", v56_dir)
print("manifest:", manifest_df.shape)
display(manifest_df[["tag", "train_start", "train_end", "test_start", "model_file"]])
print("monthly target rows:", monthly_targets_df.shape)
display(monthly_targets_df.head())

## 模型打分面板

为了分析 FN/FP，必须知道每只股票当月的模型分数，而不只是最终 target。这里直接加载 V56 导出的 pkl，对每个 OOS 月份重新打分。

In [ ]:
def prepare_model_matrix(df, feature_cols, fill_values):
    X = df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    fill = pd.Series(fill_values)
    X = X.fillna(fill).fillna(0)
    return X[feature_cols]


def score_with_bundle(df, bundle):
    feature_cols = list(bundle.get("base_feature_cols", []))
    if not feature_cols:
        raise ValueError("bundle missing base_feature_cols")
    model = bundle.get("base_model", None)
    if model is None:
        raise ValueError("bundle missing base_model")
    fill_values = dict(bundle.get("base_fill_values", {}))
    X = prepare_model_matrix(df, feature_cols, fill_values)
    iter_n = int(bundle.get("fixed_iter", bundle.get("model_iter", 0)) or 0)
    if iter_n > 0:
        score = np.asarray(model.predict(X, num_iteration=iter_n)).reshape(-1)
    else:
        score = np.asarray(model.predict(X)).reshape(-1)
    return score, feature_cols


def add_model_scores_for_one_model(df_all, manifest_row):
    bundle, model_path = load_bundle(v56_dir, manifest_row)
    test_start = pd.Timestamp(manifest_row["test_start"])
    tag = str(manifest_row["tag"])
    panel = df_all[df_all[DATE_COL] >= test_start].copy()
    score, feature_cols = score_with_bundle(panel, bundle)
    panel["model_score"] = score
    panel["model_id"] = str(manifest_row["model_id"])
    panel["tag"] = tag
    panel["train_start"] = manifest_row["train_start"]
    panel["train_end"] = manifest_row["train_end"]
    panel["test_start"] = manifest_row["test_start"]
    panel["feature_count"] = len(feature_cols)
    panel["model_path_used"] = model_path
    panel["score_rank_pct"] = panel.groupby(DATE_COL)["model_score"].rank(pct=True)
    panel["realized_rank_pct"] = panel.groupby(DATE_COL)[TARGET_COL].rank(pct=True)
    return panel, bundle, feature_cols


score_parts = []
bundle_map = {}
feature_cols_map = {}
for _, row in progress_iter(manifest_df.iterrows(), total=len(manifest_df), desc="score models"):
    score_part, bundle, feature_cols = add_model_scores_for_one_model(df_all, row)
    score_parts.append(score_part)
    bundle_map[str(row["tag"])] = bundle
    feature_cols_map[str(row["tag"])] = feature_cols
    print("scored", row["tag"], score_part.shape, "features", len(feature_cols))
    gc.collect()

score_df = pd.concat(score_parts, ignore_index=True)
print("score_df:", score_df.shape)
display(score_df[["tag", DATE_COL, STOCK_COL, TARGET_COL, "model_score", "score_rank_pct", "realized_rank_pct"]].head())

## 应用场景评估：同约束随机 top6 分位

真实策略是每月选 6 只，而不是给全市场做精确排名。因此这里构造随机组合分布：

- 同一个 OOS 月份。
- 同一个可交易股票池，即 `score_df` 当月有标签和模型分数的股票。
- 同一个行业约束选择方式：随机打乱股票后，使用和 V56 相同的 `industry_top6` 选择逻辑。

输出的 `random_percentile` 才是更贴近实盘的问题：模型这 6 只，在同约束随机组合收益分布里排第几分位？

In [ ]:
def build_industry_neutral_targets(sorted_stocks, industry_map, target_num, max_per_industry):
    known_industries = set([
        industry_map.get(stock, "UNKNOWN")
        for stock in sorted_stocks
        if industry_map.get(stock, "UNKNOWN") != "UNKNOWN"
    ])
    if len(known_industries) < 3:
        return sorted_stocks[:min(target_num, len(sorted_stocks))]

    selected = []
    industry_count = {}
    for stock in sorted_stocks:
        industry = industry_map.get(stock, "UNKNOWN")
        if industry == "UNKNOWN":
            continue
        if industry_count.get(industry, 0) == 0:
            selected.append(stock)
            industry_count[industry] = 1
            if len(selected) >= target_num:
                return selected

    for stock in sorted_stocks:
        if stock in selected:
            continue
        industry = industry_map.get(stock, "UNKNOWN")
        if industry == "UNKNOWN":
            continue
        cnt = industry_count.get(industry, 0)
        if cnt < max_per_industry:
            selected.append(stock)
            industry_count[industry] = cnt + 1
            if len(selected) >= target_num:
                return selected

    for stock in sorted_stocks:
        if stock not in selected:
            selected.append(stock)
            if len(selected) >= target_num:
                break
    return selected


def calc_drawdown_from_returns(ret_series):
    if len(ret_series) == 0:
        return np.nan
    nav = (1.0 + pd.Series(ret_series).fillna(0)).cumprod()
    dd = nav / nav.cummax() - 1.0
    return float(dd.min())


def summarize_return_series(ret_series):
    s = pd.Series(ret_series).dropna()
    if len(s) == 0:
        return {"months": 0, "cum_ret": np.nan, "mean_ret": np.nan, "win_rate": np.nan, "max_drawdown": np.nan}
    return {
        "months": int(len(s)),
        "cum_ret": float((1.0 + s).prod() - 1.0),
        "mean_ret": float(s.mean()),
        "win_rate": float((s > 0).mean()),
        "max_drawdown": calc_drawdown_from_returns(s),
    }


def calc_portfolio_return_from_map(ret_map, stocks):
    vals = []
    for stock in stocks:
        v = ret_map.get(stock, np.nan)
        if not pd.isnull(v):
            vals.append(float(v))
    return float(np.mean(vals)) if vals else np.nan


def encode_industries(industry_values):
    codes = []
    mapping = {}
    next_code = 0
    for x in industry_values:
        key = str(x) if not pd.isnull(x) else "UNKNOWN"
        if key == "UNKNOWN":
            codes.append(-1)
            continue
        if key not in mapping:
            mapping[key] = next_code
            next_code += 1
        codes.append(mapping[key])
    return np.asarray(codes, dtype=np.int32), len(mapping)


def select_random_indices_industry_neutral(perm, industry_codes, target_num, max_per_industry, known_industry_count):
    if len(perm) <= target_num or known_industry_count < 3:
        return perm[:min(target_num, len(perm))]

    selected = []
    selected_flag = np.zeros(len(industry_codes), dtype=np.bool_)
    industry_count = {}

    # First pass: one stock per known industry, matching build_industry_neutral_targets.
    for idx in perm:
        ind = int(industry_codes[idx])
        if ind < 0:
            continue
        if industry_count.get(ind, 0) == 0:
            selected.append(idx)
            selected_flag[idx] = True
            industry_count[ind] = 1
            if len(selected) >= target_num:
                return np.asarray(selected, dtype=np.int32)

    # Second pass: allow up to max_per_industry.
    for idx in perm:
        if selected_flag[idx]:
            continue
        ind = int(industry_codes[idx])
        if ind < 0:
            continue
        cnt = industry_count.get(ind, 0)
        if cnt < max_per_industry:
            selected.append(idx)
            selected_flag[idx] = True
            industry_count[ind] = cnt + 1
            if len(selected) >= target_num:
                return np.asarray(selected, dtype=np.int32)

    # Third pass: fill any remaining slot.
    for idx in perm:
        if not selected_flag[idx]:
            selected.append(idx)
            if len(selected) >= target_num:
                break
    return np.asarray(selected, dtype=np.int32)


def random_distribution_for_month_fast(month_df, sim_n, seed_key):
    rets = pd.to_numeric(month_df[TARGET_COL], errors="coerce").replace([np.inf, -np.inf], np.nan).values.astype(float)
    valid = np.isfinite(rets)
    rets = rets[valid]
    industries = month_df.loc[valid, INDUSTRY_COL].fillna("UNKNOWN").astype(str).values
    n = len(rets)
    if n == 0:
        return np.asarray([], dtype=float)
    industry_codes, known_industry_count = encode_industries(industries)
    max_per_industry = max(1, int(np.floor(TARGET_N * 0.20)))
    rng = np.random.RandomState(seed_key)
    out = np.empty(int(sim_n), dtype=float)
    base_idx = np.arange(n, dtype=np.int32)
    for i in range(int(sim_n)):
        perm = rng.permutation(base_idx)
        picked = select_random_indices_industry_neutral(
            perm,
            industry_codes,
            TARGET_N,
            max_per_industry,
            known_industry_count,
        )
        out[i] = float(np.nanmean(rets[picked])) if len(picked) else np.nan
    return out



def stable_tag_seed(tag):
    text = str(tag)
    total = 0
    for i, ch in enumerate(text):
        total += (i + 1) * ord(ch)
    return int(total % 100000)


def month_random_eval(score_df, monthly_targets_df):
    rows = []
    target_rows = monthly_targets_df.copy()
    target_rows["target_list"] = target_rows["targets"].apply(parse_targets)
    total = len(target_rows)
    iterator = progress_iter(target_rows.iterrows(), total=total, desc="random baseline months")
    for _, row in iterator:
        tag = str(row["tag"])
        dt = pd.Timestamp(row["rebalance_date"])
        month_df = score_df[(score_df["tag"].astype(str) == tag) & (score_df[DATE_COL] == dt)].copy()
        month_df = month_df.dropna(subset=[TARGET_COL, "model_score"])
        if month_df.empty:
            continue
        ret_map = dict(zip(month_df[STOCK_COL], pd.to_numeric(month_df[TARGET_COL], errors="coerce")))
        valid_stocks = set(month_df[STOCK_COL])
        target_list = [s for s in row["target_list"] if s in valid_stocks]
        if len(target_list) == 0:
            continue
        actual_top = list(month_df.sort_values(TARGET_COL, ascending=False)[STOCK_COL].head(REALIZED_TOP_N))
        actual_top20 = set(list(month_df.sort_values(TARGET_COL, ascending=False)[STOCK_COL].head(REALIZED_TOP20_N)))
        target_ret = calc_portfolio_return_from_map(ret_map, target_list)
        actual_top_ret = calc_portfolio_return_from_map(ret_map, actual_top)
        seed_key = int(pd.Timestamp(dt).strftime("%Y%m%d")) + stable_tag_seed(tag)
        rand = random_distribution_for_month_fast(month_df, RANDOM_SIM_N, seed_key)
        rand = rand[np.isfinite(rand)]
        if len(rand) == 0:
            continue
        percentile = float((rand <= target_ret).mean())
        hit_top6 = len(set(target_list).intersection(set(actual_top)))
        fp_not_top20 = len([s for s in target_list if s not in actual_top20])
        selected_rows = month_df.set_index(STOCK_COL).reindex(target_list)
        rows.append({
            "tag": tag,
            "model_id": row["model_id"],
            "rebalance_date": dt,
            "profile": TARGET_PROFILE,
            "target_count": len(target_list),
            "universe_count": int(len(month_df)),
            "target_ret": target_ret,
            "actual_top6_ret": actual_top_ret,
            "oracle_gap": actual_top_ret - target_ret,
            "random_percentile": percentile,
            "random_mean": float(np.nanmean(rand)),
            "random_median": float(np.nanmedian(rand)),
            "random_p10": float(np.nanpercentile(rand, 10)),
            "random_p90": float(np.nanpercentile(rand, 90)),
            "hit_top6": int(hit_top6),
            "fn_top6_count": int(max(0, REALIZED_TOP_N - hit_top6)),
            "fp_top6_count": int(max(0, len(target_list) - hit_top6)),
            "fp_not_top20_count": int(fp_not_top20),
            "target_avg_rank": float(selected_rows["realized_rank_pct"].mean()),
            "target_worst_rank": float(selected_rows["realized_rank_pct"].min()),
            "targets": ",".join(target_list),
            "actual_top6": ",".join(actual_top),
        })
    return pd.DataFrame(rows)


random_monthly_df = month_random_eval(score_df, monthly_targets_df)
print("random_monthly_df:", random_monthly_df.shape)
display(random_monthly_df.head())

## FP / FN 个股案例表

这里用真实应用定义：

- `FN_top6`：真实 top6 但没有被 target_top6 选中。
- `FP_top6`：被 target_top6 选中但不是真实 top6。
- `FP_not_top20`：被 target_top6 选中且真实排名不在 top20。

每条记录包含模型分数排名、真实收益排名、行业、收益、分数，方便直接抽样复盘。

In [ ]:
def build_fp_fn_cases(score_df, random_monthly_df):
    rows = []
    for _, row in progress_iter(random_monthly_df.iterrows(), total=len(random_monthly_df), desc="build FP/FN cases"):
        tag = str(row["tag"])
        dt = pd.Timestamp(row["rebalance_date"])
        month_df = score_df[(score_df["tag"].astype(str) == tag) & (score_df[DATE_COL] == dt)].copy()
        month_df = month_df.dropna(subset=[TARGET_COL, "model_score"])
        if month_df.empty:
            continue
        target_set = set(parse_targets(row["targets"]))
        actual_top6 = set(parse_targets(row["actual_top6"]))
        actual_top20 = set(list(month_df.sort_values(TARGET_COL, ascending=False)[STOCK_COL].head(REALIZED_TOP20_N)))
        case_defs = []
        for stock in sorted(actual_top6 - target_set):
            case_defs.append((stock, "FN_top6"))
        for stock in sorted(target_set - actual_top6):
            case_defs.append((stock, "FP_top6"))
            if stock not in actual_top20:
                case_defs.append((stock, "FP_not_top20"))
        mindex = month_df.set_index(STOCK_COL)
        for stock, case_type in case_defs:
            if stock not in mindex.index:
                continue
            r = mindex.loc[stock]
            rows.append({
                "tag": tag,
                "model_id": row["model_id"],
                "rebalance_date": dt,
                "case_type": case_type,
                "stock": stock,
                "industry_bucket": r.get(INDUSTRY_COL, "UNKNOWN"),
                "alpha_1m": float(r[TARGET_COL]),
                "model_score": float(r["model_score"]),
                "score_rank_pct": float(r["score_rank_pct"]),
                "realized_rank_pct": float(r["realized_rank_pct"]),
                "target_ret": row["target_ret"],
                "random_percentile": row["random_percentile"],
                "target_avg_rank": row["target_avg_rank"],
                "target_worst_rank": row["target_worst_rank"],
            })
    return pd.DataFrame(rows)


case_df = build_fp_fn_cases(score_df, random_monthly_df)
print("case_df:", case_df.shape)
display(case_df.head(20))

## 因子归因：为什么漏掉 FN，为什么误选 FP

做法是近似解释，不是假装有因果：

1. 对每个模型读取 LightGBM gain 最高的特征。
2. 在每个月横截面上计算每个特征与模型分数的相关方向。
3. 把特征转成“模型喜欢的方向”的分位：
   - 如果特征和模型分数正相关，高分位更有利。
   - 如果负相关，低分位更有利，换成 `1 - rank`。
4. FN：看真实 top6 漏选股票，在哪些高权重特征上明显弱于 target_top6 中位数。
5. FP：看误选股票，在哪些高权重特征上很强，说明模型可能被这些特征推高。

这不是 SHAP，但足够回答“模型大概因为什么没选/错选”。

In [ ]:
def get_top_gain_features(bundle, top_n):
    model = bundle.get("base_model", None)
    feature_cols = list(bundle.get("base_feature_cols", []))
    if model is None or not feature_cols:
        return []
    try:
        gain = model.feature_importance(importance_type="gain")
    except Exception:
        gain = model.feature_importance()
    imp = pd.DataFrame({"feature": feature_cols, "gain_importance": gain})
    imp = imp.sort_values("gain_importance", ascending=False)
    return list(imp["feature"].head(int(top_n)))


def feature_favorable_rank(month_df, feature, score_col="model_score"):
    s = pd.to_numeric(month_df[feature], errors="coerce").replace([np.inf, -np.inf], np.nan)
    score = pd.to_numeric(month_df[score_col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    valid = pd.DataFrame({"x": s, "score": score}).dropna()
    if valid.empty or valid["x"].nunique() <= 1:
        return pd.Series(index=month_df.index, dtype=float), np.nan
    corr = valid["x"].corr(valid["score"], method="spearman")
    raw_rank = s.rank(pct=True)
    if pd.isnull(corr):
        fav = raw_rank
    elif corr >= 0:
        fav = raw_rank
    else:
        fav = 1.0 - raw_rank
    return fav, corr


def build_feature_case_attribution(score_df, case_df):
    rows = []
    if case_df.empty:
        return pd.DataFrame()
    for tag in progress_iter(sorted(case_df["tag"].astype(str).unique()), total=case_df["tag"].astype(str).nunique(), desc="attribute tags"):
        bundle = bundle_map.get(tag)
        if bundle is None:
            continue
        top_features = get_top_gain_features(bundle, ATTR_TOP_FEATURE_N)
        tag_cases = case_df[case_df["tag"].astype(str) == tag].copy()
        month_groups = list(tag_cases.groupby("rebalance_date"))
        for dt, month_cases in progress_iter(month_groups, total=len(month_groups), desc="attribute months %s" % tag):
            dt = pd.Timestamp(dt)
            month_df = score_df[(score_df["tag"].astype(str) == tag) & (score_df[DATE_COL] == dt)].copy()
            month_df = month_df.dropna(subset=[TARGET_COL, "model_score"])
            if month_df.empty:
                continue
            targets = []
            row_match = random_monthly_df[(random_monthly_df["tag"].astype(str) == tag) & (random_monthly_df["rebalance_date"] == dt)]
            if not row_match.empty:
                targets = parse_targets(row_match.iloc[0]["targets"])
            selected_df = month_df[month_df[STOCK_COL].isin(targets)].copy()
            mindex = month_df.set_index(STOCK_COL)
            selected_index = selected_df.set_index(STOCK_COL)
            for feature in top_features:
                if feature not in month_df.columns:
                    continue
                fav, corr = feature_favorable_rank(month_df, feature)
                month_df["_fav"] = fav
                selected_fav_median = float(month_df[month_df[STOCK_COL].isin(targets)]["_fav"].median()) if len(targets) else np.nan
                universe_fav_median = float(month_df["_fav"].median())
                fav_by_stock = month_df.set_index(STOCK_COL)["_fav"]
                val_by_stock = month_df.set_index(STOCK_COL)[feature]
                for _, case in month_cases.iterrows():
                    stock = case["stock"]
                    if stock not in fav_by_stock.index:
                        continue
                    case_fav = fav_by_stock.loc[stock]
                    if pd.isnull(case_fav):
                        continue
                    gap_vs_selected = case_fav - selected_fav_median if not pd.isnull(selected_fav_median) else np.nan
                    gap_vs_universe = case_fav - universe_fav_median if not pd.isnull(universe_fav_median) else np.nan
                    rows.append({
                        "tag": tag,
                        "rebalance_date": dt,
                        "case_type": case["case_type"],
                        "stock": stock,
                        "feature": feature,
                        "feature_value": val_by_stock.loc[stock],
                        "feature_score_corr": corr,
                        "case_favorable_rank": case_fav,
                        "selected_median_favorable_rank": selected_fav_median,
                        "universe_median_favorable_rank": universe_fav_median,
                        "gap_vs_selected": gap_vs_selected,
                        "gap_vs_universe": gap_vs_universe,
                        "model_score_rank": case["score_rank_pct"],
                        "realized_rank": case["realized_rank_pct"],
                    })
    return pd.DataFrame(rows)


feature_case_df = build_feature_case_attribution(score_df, case_df)
print("feature_case_df:", feature_case_df.shape)
display(feature_case_df.head(20))

## 汇总：随机分位、FN/FP 频率、因子原因

这几个表是 V57 的主要输出：

- `random_summary_df`：每个模型相对随机组合的真实应用分位。
- `case_summary_df`：FN/FP 频率。
- `feature_failure_summary_df`：错例中反复出现的高权重特征原因。

In [ ]:
def build_random_summary(random_monthly_df):
    rows = []
    for tag, gdf in random_monthly_df.groupby("tag"):
        ret_sum = summarize_return_series(gdf["target_ret"])
        random_mean_sum = summarize_return_series(gdf["random_mean"])
        rows.append({
            "tag": tag,
            "months": int(len(gdf)),
            "target_cum_ret": ret_sum["cum_ret"],
            "target_mean_ret": ret_sum["mean_ret"],
            "target_win_rate": ret_sum["win_rate"],
            "target_max_drawdown": ret_sum["max_drawdown"],
            "random_mean_cum_ret": random_mean_sum["cum_ret"],
            "random_mean_ret": random_mean_sum["mean_ret"],
            "mean_random_percentile": float(gdf["random_percentile"].mean()),
            "median_random_percentile": float(gdf["random_percentile"].median()),
            "pct_months_above_random_median": float((gdf["random_percentile"] > 0.50).mean()),
            "pct_months_top_quartile": float((gdf["random_percentile"] > 0.75).mean()),
            "pct_months_bottom_quartile": float((gdf["random_percentile"] < 0.25).mean()),
            "avg_hit_top6": float(gdf["hit_top6"].mean()),
            "avg_fn_top6_count": float(gdf["fn_top6_count"].mean()),
            "avg_fp_top6_count": float(gdf["fp_top6_count"].mean()),
            "avg_fp_not_top20_count": float(gdf["fp_not_top20_count"].mean()),
            "avg_oracle_gap": float(gdf["oracle_gap"].mean()),
            "avg_target_rank": float(gdf["target_avg_rank"].mean()),
        })
    return pd.DataFrame(rows)


def build_case_summary(case_df):
    if case_df.empty:
        return pd.DataFrame()
    rows = []
    for keys, gdf in case_df.groupby(["tag", "case_type"]):
        tag, case_type = keys
        rows.append({
            "tag": tag,
            "case_type": case_type,
            "cases": int(len(gdf)),
            "months": int(gdf["rebalance_date"].nunique()),
            "mean_alpha": float(gdf["alpha_1m"].mean()),
            "mean_score_rank": float(gdf["score_rank_pct"].mean()),
            "mean_realized_rank": float(gdf["realized_rank_pct"].mean()),
        })
    return pd.DataFrame(rows)


def build_feature_failure_summary(feature_case_df):
    if feature_case_df.empty:
        return pd.DataFrame()
    parts = []
    for keys, gdf in feature_case_df.groupby(["tag", "case_type", "feature"]):
        tag, case_type, feature = keys
        if case_type.startswith("FN"):
            # FN 关注：哪些特征相对已选组合明显偏弱。
            severity = -gdf["gap_vs_selected"]
        else:
            # FP 关注：哪些特征很被模型喜欢，却没有转成真实收益。
            severity = gdf["case_favorable_rank"]
        parts.append({
            "tag": tag,
            "case_type": case_type,
            "feature": feature,
            "cases": int(len(gdf)),
            "avg_case_favorable_rank": float(gdf["case_favorable_rank"].mean()),
            "avg_gap_vs_selected": float(gdf["gap_vs_selected"].mean()),
            "avg_gap_vs_universe": float(gdf["gap_vs_universe"].mean()),
            "avg_feature_score_corr": float(gdf["feature_score_corr"].mean()),
            "severity_score": float(severity.mean()),
        })
    out = pd.DataFrame(parts)
    return out.sort_values(["tag", "case_type", "severity_score"], ascending=[True, True, False])


random_summary_df = build_random_summary(random_monthly_df)
case_summary_df = build_case_summary(case_df)
feature_failure_summary_df = build_feature_failure_summary(feature_case_df)

print("random summary")
display(random_summary_df)
print("case summary")
display(case_summary_df)
print("feature failure summary")
display(feature_failure_summary_df.head(80))

## 导出结果

重点看：

- `v57_random_monthly.csv`：每月模型 top6 相对随机组合的分位。
- `v57_random_summary.csv`：每个模型的应用场景汇总。
- `v57_fp_fn_cases.csv`：所有 FN/FP 个股案例。
- `v57_fp_fn_feature_attribution.csv`：个股-特征级别归因。
- `v57_feature_failure_summary.csv`：高频 FN/FP 因子原因。

In [ ]:
random_monthly_path = os.path.join(OUT_DIR, "v57_random_monthly.csv")
random_summary_path = os.path.join(OUT_DIR, "v57_random_summary.csv")
case_path = os.path.join(OUT_DIR, "v57_fp_fn_cases.csv")
feature_case_path = os.path.join(OUT_DIR, "v57_fp_fn_feature_attribution.csv")
feature_summary_path = os.path.join(OUT_DIR, "v57_feature_failure_summary.csv")

random_monthly_df.to_csv(random_monthly_path, index=False)
random_summary_df.to_csv(random_summary_path, index=False)
case_df.to_csv(case_path, index=False)
feature_case_df.to_csv(feature_case_path, index=False)
feature_failure_summary_df.to_csv(feature_summary_path, index=False)

print("saved:")
for p in [random_monthly_path, random_summary_path, case_path, feature_case_path, feature_summary_path]:
    print(" ", p)

## 结论模板

运行后按这个顺序读结果：

1. `v57_random_summary.csv`：如果 `mean_random_percentile` 长期只是 0.50 附近，说明模型并不明显优于随机 top6；如果稳定 > 0.60，才更能说明实用选股有效。
2. `v57_random_monthly.csv`：观察是否只靠少数月份贡献；重点看 `random_percentile < 0.25` 的月份是否连续出现。
3. `v57_fp_fn_cases.csv`：抽最近几个月的 FN/FP 看是否有系统性行业/板块/风格偏差。
4. `v57_feature_failure_summary.csv`：看 FN 是否总是因为某几个高权重特征被压低，FP 是否总是被某几个高权重特征误推高。

V57 的目标不是证明模型强，而是把“强在哪里、漏在哪里、错在哪里”讲清楚。